# Skill Passport: training, explanations and ONNX export

Run top to bottom in a fresh Python 3.12 Colab runtime. Start with the explicitly synthetic software test. Real training requires the approved Supabase schema, explicit employee consent and independently assessed labels. Do not share service keys or private records in notebook outputs. This notebook was prepared and its Python modules tested locally; it has not been run in your personal Colab account.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, os, sys, subprocess
uploaded = files.upload()  # Select Skill_Passport_ML_Pipeline.zip
name = next(iter(uploaded))
root = Path('/content/skill-passport').resolve()
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    for info in archive.infolist():
        target = (root / info.filename).resolve()
        if not target.is_relative_to(root) or ((info.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError('Unsafe archive member')
    archive.extractall(root)
os.chdir(root)
assert Path('ml/requirements-training.txt').exists(), 'Upload the ML ZIP containing ml/'
print('Source extracted. No secrets loaded.')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ml/requirements-training.txt'], check=True)


## 1. Verify the software with synthetic data

These results test code, not real-world employee readiness. The synthetic bundle is prohibited from production publication.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'ml/tests', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'ml.train_and_explain', '--demo', '--trials', '2', '--output', 'ml/artifacts/demo'], check=True)
subprocess.run([sys.executable, '-m', 'ml.pipeline_runner', '--demo', '--bundle', 'ml/artifacts/demo', '--output', 'demo_predictions.json'], check=True)


In [ ]:
import json
demo = json.loads(Path('demo_predictions.json').read_text())
print('SYNTHETIC DEMONSTRATION ONLY')
for row in demo['scored']:
    print({'score': row['role_readiness_pred'], 'explanation': row['explanation']['summary']})


## 2. Connect your real Supabase project

First apply the reviewed migration after approval. In Colab Secrets add SUPABASE_URL and SUPABASE_SERVICE_KEY, enable notebook access, and keep this notebook private. Never paste or print the key.

Collect real snapshots before later independent job-skill assessments. Follow docs/BEGINNER_GUIDE.md and ml/prepare_training_data.py. At least 100 consented examples from 30 people are required by the software; broader validation is still necessary. No real records are generated by this notebook.

In [ ]:
RUN_REAL = False  # Change only after the database and real labels are ready.
if RUN_REAL:
    from google.colab import userdata
    # Subprocesses do not inherit Colab's interactive secrets API, so pass secrets via environment.
    real_env = dict(os.environ)
    real_env['SUPABASE_URL'] = userdata.get('SUPABASE_URL')
    real_env['SUPABASE_SERVICE_KEY'] = userdata.get('SUPABASE_SERVICE_KEY')
    subprocess.run([sys.executable, '-m', 'ml.train_and_explain', '--trials', '20',
                    '--output', 'ml/artifacts/real'], env=real_env, check=True)
else:
    print('Real training is disabled. Demo did not write to Supabase.')


## 3. Review the real model

Inspect holdout MAE/RMSE versus the baseline, grouped split, ONNX error and the model card. Conduct time-based and subgroup validation on real evidence before employment use. Passing a test or beating a baseline is not sufficient validation.

In [ ]:
if RUN_REAL:
    manifest = json.loads(Path('ml/artifacts/real/manifest.json').read_text())
    print(json.dumps({k: manifest[k] for k in ['model_version', 'data_source', 'metrics', 'limitations']}, indent=2))


## 4. Upload a reviewed real bundle — inactive

Upload does not activate the model. Activation is an explicit database-owner step shown in the guide. The private models bucket is read only by the server. Synthetic upload is rejected.

In [ ]:
UPLOAD_REVIEWED_BUNDLE = False
if RUN_REAL and UPLOAD_REVIEWED_BUNDLE:
    from ml.data_loader import SupabaseLoader
    from ml.train_and_explain import upload_bundle
    upload_bundle('ml/artifacts/real', SupabaseLoader())
    print('Real bundle uploaded and registered INACTIVE. Review before activation.')


## 5. Download demo artifacts if needed

The demo bundle contains synthetic artifacts only. Do not download/share private employee snapshots or service keys with source code.

In [ ]:
import shutil
shutil.make_archive('/content/skill-passport-demo-model', 'zip', 'ml/artifacts/demo')
files.download('/content/skill-passport-demo-model.zip')
